# Fake Job Posting Detection - Google Colab Presentation Notebook

## 1. Setup & Dependencies

This opening section prepares the Google Colab runtime exactly like a reproducible engineering environment. It clones the project repository, moves the notebook runtime into the repository directory, and installs the Python dependencies declared by the project. Keeping setup commands in one cell makes the notebook easy to rerun during a college presentation or viva.

In [ ]:
%cd /content
!git clone https://github.com/Anshul-Bhardwaj-21/fake-job-detection
%cd /content/fake-job-detection
!pip install -q -r requirements.txt

## 2. Imports

This cell consolidates all core libraries used across the modular project files. The imports cover data handling, text vectorization, supervised machine learning, model calibration, association rule mining, K-Means clustering, visualization, and model persistence. Consolidating imports here keeps the rest of the notebook focused on the project workflow rather than scattered dependency setup.

In [ ]:
import json
import re
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import sparse

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.cluster import MiniBatchKMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    silhouette_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

## 3. Data Loading

The original project expects the Real or Fake Job Posting dataset in `data/raw/fake_job_postings.csv`, while also supporting an additional Indian synthetic dataset. This cell recreates the repository's dataset-loading behavior and adds a Colab-safe fallback: if no local CSV is present after cloning, the notebook downloads the normalized Hugging Face dataset used by the project. The cell then prints the dataset sources, previews the first records, and displays schema information for transparency.

In [ ]:
ROOT_DIR = Path.cwd()
DATA_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
REPORT_DIR = ROOT_DIR / "reports" / "figures"
MODEL_DIR = ROOT_DIR / "models"

KAGGLE_DATASET = DATA_DIR / "fake_job_postings.csv"
SAMPLE_DATASET = DATA_DIR / "sample_fake_job_postings.csv"
INDIAN_SYNTHETIC_DATASET = DATA_DIR / "indian_fake_job_postings_synthetic.csv"
PROCESSED_DATASET = PROCESSED_DIR / "processed_jobs.csv"
MODEL_PATH = MODEL_DIR / "fake_job_model.pkl"
TFIDF_PATH = MODEL_DIR / "tfidf_vectorizer.pkl"
LABEL_ENCODERS_PATH = MODEL_DIR / "label_encoders.pkl"
FEATURE_CONFIG_PATH = MODEL_DIR / "feature_config.pkl"
METRICS_PATH = MODEL_DIR / "metrics.json"

TEXT_COLUMNS = ["title", "company_profile", "description", "requirements", "benefits", "extra_text"]
CATEGORICAL_COLUMNS = ["employment_type", "required_experience", "required_education", "industry", "function"]
BINARY_COLUMNS = ["telecommuting", "has_company_logo", "has_questions"]
TARGET_COLUMN = "fraudulent"

INDIAN_SYNTHETIC_URL = (
    "https://huggingface.co/api/datasets/"
    "Aioshi/smolified-fakejob/parquet/default/train/0.parquet"
)


def normalize_schema(data: pd.DataFrame) -> pd.DataFrame:
    """Ensure every supported dataset has the columns expected by the project."""
    data = data.copy()

    for col in TEXT_COLUMNS:
        if col not in data.columns:
            data[col] = ""

    for col in CATEGORICAL_COLUMNS:
        if col not in data.columns:
            data[col] = "Unknown"

    for col in BINARY_COLUMNS:
        if col not in data.columns:
            data[col] = 0

    if TARGET_COLUMN not in data.columns:
        raise ValueError(f"Dataset must contain target column: {TARGET_COLUMN}")

    return data


def normalize_indian_synthetic_dataset(raw_data: pd.DataFrame) -> pd.DataFrame:
    """Convert the Hugging Face Indian synthetic fake-job dataset into project schema."""
    labels = raw_data["assistant"].fillna("").str.extract(
        r"Classification:\s*(Fake|Real)", flags=re.IGNORECASE, expand=False
    )
    valid = labels.str.lower().isin(["fake", "real"])
    job_text = raw_data.loc[valid, "user"].fillna("").astype(str)
    job_text = job_text.str.replace(r"^\s*Job Posting:\s*", "", regex=True)

    normalized = pd.DataFrame({
        "title": "Synthetic Indian Job Posting",
        "company_profile": "",
        "description": job_text,
        "requirements": "",
        "benefits": "",
        "extra_text": "",
        "employment_type": "Unknown",
        "required_experience": "Unknown",
        "required_education": "Unknown",
        "industry": "Unknown",
        "function": "Unknown",
        "telecommuting": 0,
        "has_company_logo": 0,
        "has_questions": 0,
        "fraudulent": labels.loc[valid].str.lower().map({"real": 0, "fake": 1}).astype(int),
    })

    return normalized.reset_index(drop=True)


def download_indian_synthetic_dataset(path: Path = INDIAN_SYNTHETIC_DATASET) -> Path:
    """Download and save the additional Indian fake-job dataset as CSV."""
    raw_data = pd.read_parquet(INDIAN_SYNTHETIC_URL)
    normalized = normalize_indian_synthetic_dataset(raw_data)
    path.parent.mkdir(parents=True, exist_ok=True)
    normalized.to_csv(path, index=False)
    return path


def load_dataset(path: Path) -> pd.DataFrame:
    data = pd.read_csv(path)
    return normalize_schema(data)


def resolve_primary_dataset_path() -> Path:
    """Find the best available dataset for Colab execution."""
    if KAGGLE_DATASET.exists():
        return KAGGLE_DATASET
    if SAMPLE_DATASET.exists():
        return SAMPLE_DATASET
    if INDIAN_SYNTHETIC_DATASET.exists():
        return INDIAN_SYNTHETIC_DATASET

    print("No local dataset found. Downloading the Hugging Face fallback dataset...")
    try:
        return download_indian_synthetic_dataset()
    except Exception as exc:
        raise FileNotFoundError(
            "No dataset was found. Upload fake_job_postings.csv to data/raw/ "
            "or configure Kaggle credentials and run scripts/download_dataset.py."
        ) from exc


def load_training_dataset(include_auxiliary: bool = True):
    """Load the primary training data plus any available auxiliary dataset."""
    primary_path = resolve_primary_dataset_path()
    frames = [load_dataset(primary_path)]
    sources = [{"name": "primary", "path": str(primary_path), "rows": len(frames[0])}]

    if include_auxiliary and primary_path != INDIAN_SYNTHETIC_DATASET and INDIAN_SYNTHETIC_DATASET.exists():
        auxiliary = load_dataset(INDIAN_SYNTHETIC_DATASET)
        frames.append(auxiliary)
        sources.append({
            "name": "indian_synthetic_huggingface",
            "path": str(INDIAN_SYNTHETIC_DATASET),
            "rows": len(auxiliary),
        })

    combined = pd.concat(frames, ignore_index=True)
    dedupe_columns = [*TEXT_COLUMNS, *CATEGORICAL_COLUMNS, *BINARY_COLUMNS, TARGET_COLUMN]
    combined = combined.drop_duplicates(subset=[col for col in dedupe_columns if col in combined.columns])

    if primary_path == SAMPLE_DATASET:
        sources[0]["warning"] = "sample dataset only; use the full Kaggle dataset for final training"

    return combined, sources


raw_data, dataset_sources = load_training_dataset()

print("Using training datasets:")
for source in dataset_sources:
    print(f"- {source['name']}: {source['rows']:,} rows ({source['path']})")
    if "warning" in source:
        print(f"  Warning: {source['warning']}")

print(f"\nCombined dataset shape: {raw_data.shape}")
display(raw_data.head())
raw_data.info()
display(raw_data[TARGET_COLUMN].value_counts().rename(index={0: "Real", 1: "Fake"}).to_frame("count"))

## 4. Data Preprocessing & Feature Engineering

This section ports the preprocessing logic from the project into notebook form. It normalizes missing values, converts binary and target columns to numeric form, removes duplicate rows, builds a combined text field, cleans text with regular expressions, and engineers risk-oriented features such as missing company profile, missing salary information, suspicious keyword counts, payment or deposit language, urgency terms, risky contact channels, and sensitive-document requests. It also defines the TF-IDF, one-hot encoding, and numeric-scaling helpers used later for model training and live prediction.

In [ ]:
SUSPICIOUS_KEYWORDS = [
    "registration fee", "joining fee", "processing fee", "document fee",
    "verification fee", "security deposit", "refundable deposit", "urgent hiring",
    "immediate joining", "no experience", "daily payment", "daily income",
    "guaranteed income", "guaranteed job", "100% placement", "easy money",
    "work from home", "work from mobile", "limited seats", "limited slots",
    "training fee", "pay first", "no interview", "whatsapp", "telegram",
    "aadhar", "aadhaar", "bank details",
]

FEE_KEYWORDS = [
    "registration fee", "joining fee", "processing fee", "document fee",
    "documentation fee", "verification fee", "security deposit", "refundable deposit",
    "training fee", "pay first", "deposit", "fee required", "starter kit",
]

URGENCY_KEYWORDS = [
    "urgent hiring", "immediate joining", "apply immediately", "limited seats",
    "limited slots", "selected today", "instant selection", "fast selection",
    "no interview", "guaranteed job", "100% placement",
]

CONTACT_RISK_KEYWORDS = [
    "whatsapp", "telegram", "dm now", "personal mobile", "send resume to mobile",
    "gmail.com", "protonmail", "non-official link",
]

SENSITIVE_INFO_KEYWORDS = [
    "aadhar", "aadhaar", "pan card", "bank details", "account number",
    "ifsc", "upi", "id proof", "passport copy",
]

SALARY_KEYWORDS = [
    "salary", "pay", "compensation", "ctc", "lpa", "stipend",
    "per month", "per annum", "rs", "rupees",
]

NUMERIC_COLUMNS = [
    "telecommuting", "has_company_logo", "has_questions",
    "description_length", "requirements_length", "company_profile_length",
    "suspicious_keyword_count", "fee_keyword_count", "urgency_keyword_count",
    "contact_risk_keyword_count", "sensitive_info_keyword_count",
    "profile_missing", "salary_missing",
]


def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def count_keywords(text: str, keywords: list[str]) -> int:
    text = str(text).lower()
    return sum(1 for keyword in keywords if keyword in text)


def count_suspicious_keywords(text: str) -> int:
    return count_keywords(text, SUSPICIOUS_KEYWORDS)


def prepare_dataframe(data: pd.DataFrame) -> pd.DataFrame:
    data = normalize_schema(data)

    for col in TEXT_COLUMNS:
        data[col] = data[col].fillna("").astype(str)

    for col in CATEGORICAL_COLUMNS:
        data[col] = data[col].fillna("Unknown").astype(str)

    for col in BINARY_COLUMNS:
        data[col] = pd.to_numeric(data[col], errors="coerce").fillna(0).astype(int)

    data[TARGET_COLUMN] = pd.to_numeric(data[TARGET_COLUMN], errors="coerce").fillna(0).astype(int)
    data = data.drop_duplicates()

    data["combined_text"] = data[TEXT_COLUMNS].agg(" ".join, axis=1)
    data["clean_text"] = data["combined_text"].apply(clean_text)

    data["description_length"] = data["description"].apply(lambda x: len(str(x).split()))
    data["requirements_length"] = data["requirements"].apply(lambda x: len(str(x).split()))
    data["company_profile_length"] = data["company_profile"].apply(lambda x: len(str(x).split()))

    data["profile_missing"] = data["company_profile"].apply(lambda x: 1 if len(str(x).strip()) == 0 else 0)
    data["salary_missing"] = data["combined_text"].apply(lambda x: 0 if count_keywords(x, SALARY_KEYWORDS) > 0 else 1)

    data["suspicious_keyword_count"] = data["combined_text"].apply(count_suspicious_keywords)
    data["fee_keyword_count"] = data["combined_text"].apply(lambda x: count_keywords(x, FEE_KEYWORDS))
    data["urgency_keyword_count"] = data["combined_text"].apply(lambda x: count_keywords(x, URGENCY_KEYWORDS))
    data["contact_risk_keyword_count"] = data["combined_text"].apply(lambda x: count_keywords(x, CONTACT_RISK_KEYWORDS))
    data["sensitive_info_keyword_count"] = data["combined_text"].apply(lambda x: count_keywords(x, SENSITIVE_INFO_KEYWORDS))

    return data


def make_document_style_records(data: pd.DataFrame) -> pd.DataFrame:
    """Collapse structured rows into one text block to mimic PDFs and URLs."""
    document_rows = data.copy()
    text_columns = [col for col in TEXT_COLUMNS if col in document_rows.columns]
    document_rows["description"] = document_rows[text_columns].fillna("").astype(str).agg(" ".join, axis=1)

    for col in ("company_profile", "requirements", "benefits", "extra_text"):
        document_rows[col] = ""

    document_rows["has_company_logo"] = 0
    document_rows["has_questions"] = 0
    return document_rows


def build_user_record(
    title,
    company_profile,
    description,
    requirements,
    benefits,
    employment_type="Unknown",
    required_experience="Unknown",
    required_education="Unknown",
    industry="Unknown",
    function="Unknown",
    telecommuting=0,
    has_company_logo=0,
    has_questions=0,
    extra_text="",
):
    return pd.DataFrame([{
        "title": title,
        "company_profile": company_profile,
        "description": description,
        "requirements": requirements,
        "benefits": benefits,
        "extra_text": extra_text,
        "employment_type": employment_type,
        "required_experience": required_experience,
        "required_education": required_education,
        "industry": industry,
        "function": function,
        "telecommuting": int(telecommuting),
        "has_company_logo": int(has_company_logo),
        "has_questions": int(has_questions),
        "fraudulent": 0,
    }])


def fit_feature_transformers(data, max_features=8000):
    """Fit TF-IDF, one-hot encoder, scaler, and return the combined sparse matrix."""
    tfidf = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2), min_df=2)
    X_text = tfidf.fit_transform(data["clean_text"])

    ohe = OneHotEncoder(handle_unknown="ignore")
    X_cat = ohe.fit_transform(data[CATEGORICAL_COLUMNS])

    scaler = StandardScaler(with_mean=False)
    X_num = scaler.fit_transform(data[NUMERIC_COLUMNS])

    X_combined = sparse.hstack([X_text, X_cat, X_num])
    return tfidf, ohe, scaler, X_combined


def transform_features(data, tfidf, ohe, scaler):
    """Transform prepared rows using the already fitted feature transformers."""
    X_text = tfidf.transform(data["clean_text"])
    X_cat = ohe.transform(data[CATEGORICAL_COLUMNS])
    X_num = scaler.transform(data[NUMERIC_COLUMNS])
    return sparse.hstack([X_text, X_cat, X_num])


prepared = prepare_dataframe(raw_data)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
prepared.to_csv(PROCESSED_DATASET, index=False)

print(f"Prepared dataset shape: {prepared.shape}")
print(f"Processed data saved to: {PROCESSED_DATASET}")
display(prepared[[
    "title", "clean_text", "description_length", "requirements_length",
    "profile_missing", "salary_missing", "suspicious_keyword_count", TARGET_COLUMN,
]].head())

## 5. Model Training

This cell follows the training strategy from the modular project. First, it creates an honest train-test split with stratification on the target label. Next, it augments the training data with document-style versions of each job post so the classifier can handle both structured form inputs and raw document text. The cell then fits TF-IDF, one-hot encoding, and numeric scaling on the training split, trains the candidate classifiers, ranks them by F1-score and recall, and runs the unsupervised data-mining modules: Apriori association rules and MiniBatch K-Means clustering.

In [ ]:
train_df, test_df = train_test_split(
    prepared,
    test_size=0.2,
    random_state=42,
    stratify=prepared[TARGET_COLUMN],
)

train_augmented = prepare_dataframe(
    pd.concat([train_df, make_document_style_records(train_df)], ignore_index=True)
)

tfidf, ohe, scaler, X_train = fit_feature_transformers(train_augmented)
X_test = transform_features(test_df, tfidf, ohe, scaler)
y_train = train_augmented[TARGET_COLUMN]
y_test = test_df[TARGET_COLUMN]

candidate_models = {
    "Complement Naive Bayes": ComplementNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=120, random_state=42, class_weight="balanced"),
    "Linear SVM": LinearSVC(class_weight="balanced", random_state=42, max_iter=10000),
    "Calibrated Linear SVM": CalibratedClassifierCV(
        estimator=LinearSVC(class_weight="balanced", random_state=42, max_iter=10000),
        cv=3,
    ),
}

results = []
trained = {}

for name, clf in candidate_models.items():
    try:
        clf.fit(X_train, y_train)
        preds = clf.predict(X_test)

        result = {
            "model": name,
            "accuracy": float(accuracy_score(y_test, preds)),
            "precision": float(precision_score(y_test, preds, zero_division=0)),
            "recall": float(recall_score(y_test, preds, zero_division=0)),
            "f1_score": float(f1_score(y_test, preds, zero_division=0)),
        }
        results.append(result)
        trained[name] = clf
        print(f"Trained {name}: F1={result['f1_score']:.4f}, Recall={result['recall']:.4f}")
    except Exception as exc:
        print(f"Error training {name}: {exc}")

results_df = pd.DataFrame(results).sort_values(by=["f1_score", "recall"], ascending=False)
best_name = results_df.iloc[0]["model"]
best_model = trained[best_name]

print(f"\nBest holdout model: {best_name}")
display(results_df)


def make_transaction_items(row):
    items = []
    if row.get("telecommuting", 0) == 1:
        items.append("Remote Job")
    if row.get("has_company_logo", 0) == 0:
        items.append("Missing Logo")
    if row.get("has_questions", 0) == 0:
        items.append("No Screening Questions")
    if row.get("profile_missing", 0) == 1:
        items.append("Missing Company Profile")
    if row.get("salary_missing", 0) == 1:
        items.append("Salary Not Mentioned")
    if row.get("suspicious_keyword_count", 0) >= 1:
        items.append("Suspicious Keywords Present")
    if row.get("fee_keyword_count", 0) >= 1:
        items.append("Payment Or Deposit Request")
    if row.get("urgency_keyword_count", 0) >= 1:
        items.append("Urgency Or Guaranteed Selection")
    if row.get("contact_risk_keyword_count", 0) >= 1:
        items.append("Risky Contact Channel")
    if row.get("sensitive_info_keyword_count", 0) >= 1:
        items.append("Sensitive Info Requested")
    if row.get("fraudulent", 0) == 1:
        items.append("Fraudulent")
    else:
        items.append("Genuine")
    return items


def generate_simple_association_rules(data, min_support=0.005, min_confidence=0.3):
    """Generate association rules using the mlxtend Apriori algorithm."""
    transactions = [make_transaction_items(row) for _, row in data.iterrows()]

    te = TransactionEncoder()
    te_ary = te.fit_transform(transactions)
    encoded_transactions = pd.DataFrame(te_ary, columns=te.columns_)

    frequent_itemsets = apriori(encoded_transactions, min_support=min_support, use_colnames=True)

    if frequent_itemsets.empty:
        print("No frequent itemsets found with current min_support. Try lowering min_support.")
        rules_df = pd.DataFrame()
    else:
        rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_confidence)

        if rules.empty:
            print("No association rules found with current parameters.")
            rules_df = pd.DataFrame()
        else:
            fraud_rules = rules[rules["consequents"].apply(lambda x: "Fraudulent" in x)]

            if not fraud_rules.empty:
                rules_df = fraud_rules[["antecedents", "consequents", "support", "confidence", "lift"]].copy()
                rules_df["antecedents"] = rules_df["antecedents"].apply(lambda x: ", ".join(list(x)))
                rules_df["consequents"] = rules_df["consequents"].apply(lambda x: ", ".join(list(x)))
                rules_df = rules_df.sort_values(by=["lift", "confidence"], ascending=False)
            else:
                print("No rules with 'Fraudulent' as consequent found.")
                rules_df = pd.DataFrame()

    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    rules_df.to_csv(REPORT_DIR / "association_rules.csv", index=False)
    return rules_df


def run_clustering(data, max_k=8):
    REPORT_DIR.mkdir(parents=True, exist_ok=True)

    vectorizer = TfidfVectorizer(max_features=2000, ngram_range=(1, 2))
    X = vectorizer.fit_transform(data["clean_text"])

    inertias = []
    silhouette_scores = []
    k_values = list(range(2, max_k + 1))

    for k in k_values:
        model = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=1000, n_init=10)
        labels = model.fit_predict(X)
        inertias.append(model.inertia_)
        try:
            silhouette_scores.append(float(silhouette_score(X, labels, sample_size=1000)))
        except Exception:
            silhouette_scores.append(0.0)

    best_index = max(range(len(silhouette_scores)), key=lambda i: silhouette_scores[i])
    best_k = k_values[best_index]
    final_model = MiniBatchKMeans(n_clusters=best_k, random_state=42, batch_size=1000, n_init=10)
    clustered_data = data.copy()
    clustered_data["cluster"] = final_model.fit_predict(X)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(k_values, inertias, marker="o", linestyle="-", color="b")
    ax.set_xlabel("Number of Clusters (k)")
    ax.set_ylabel("WCSS / Inertia")
    ax.set_title("Elbow Method for Optimal k")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(REPORT_DIR / "clustering_elbow.png", dpi=160)
    plt.show()

    summary = clustered_data.groupby("cluster").agg(
        total_jobs=("cluster", "count"),
        fake_jobs=("fraudulent", "sum"),
        avg_suspicious_keywords=("suspicious_keyword_count", "mean"),
        avg_fee_keywords=("fee_keyword_count", "mean"),
        avg_urgency_keywords=("urgency_keyword_count", "mean"),
        avg_contact_risk_keywords=("contact_risk_keyword_count", "mean"),
        avg_sensitive_info_keywords=("sensitive_info_keyword_count", "mean"),
        avg_profile_missing=("profile_missing", "mean"),
        avg_salary_missing=("salary_missing", "mean"),
    ).reset_index()
    summary["fake_ratio"] = summary["fake_jobs"] / summary["total_jobs"]
    summary.to_csv(REPORT_DIR / "cluster_summary.csv", index=False)

    return {
        "best_k": best_k,
        "silhouette_scores": dict(zip(k_values, silhouette_scores)),
        "cluster_summary": summary.to_dict(orient="records"),
    }


print("\nGenerating Apriori association rules...")
rules_df = generate_simple_association_rules(prepared)
display(rules_df.head(10) if not rules_df.empty else pd.DataFrame({"message": ["No rules generated."]}))

print("\nRunning K-Means clustering...")
clustering_result = run_clustering(prepared)
display(pd.DataFrame(clustering_result["cluster_summary"]))

## 6. Model Evaluation

This evaluation section reports the standard classification metrics required for academic assessment: accuracy, precision, recall, F1-score, and the confusion matrix. It evaluates the best model on the untouched holdout set, also checks a document-style holdout representation, saves report artifacts, and then retrains the selected best model on all available data for deployment-style live prediction in the final cell.

In [ ]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

best_preds = best_model.predict(X_test)
cm = confusion_matrix(y_test, best_preds)

accuracy = accuracy_score(y_test, best_preds)
precision = precision_score(y_test, best_preds, zero_division=0)
recall = recall_score(y_test, best_preds, zero_division=0)
f1 = f1_score(y_test, best_preds, zero_division=0)

print(f"Best Model: {best_name}")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, best_preds, target_names=["Real", "Fake"], zero_division=0))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    ax=ax,
    xticklabels=["Real", "Fake"],
    yticklabels=["Real", "Fake"],
)
ax.set_title(f"Confusion Matrix - {best_name}")
ax.set_ylabel("True Label")
ax.set_xlabel("Predicted Label")
plt.tight_layout()
plt.savefig(REPORT_DIR / "confusion_matrix.png", dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
results_df.set_index("model")[["accuracy", "precision", "recall", "f1_score"]].plot(kind="bar", ax=ax)
ax.set_title("Model Comparison")
ax.set_ylabel("Score")
ax.set_xlabel("Model")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(REPORT_DIR / "model_comparison.png", dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
prepared[TARGET_COLUMN].value_counts().sort_index().plot(kind="bar", ax=ax, color=["skyblue", "salmon"])
ax.set_xticklabels(["Real", "Fake"], rotation=0)
ax.set_title("Class Distribution")
ax.set_ylabel("Count")
ax.set_xlabel("Job Type")
plt.tight_layout()
plt.savefig(REPORT_DIR / "class_distribution.png", dpi=160)
plt.show()

doc_test_df = prepare_dataframe(make_document_style_records(test_df))
X_doc_test = transform_features(doc_test_df, tfidf, ohe, scaler)
doc_test_preds = best_model.predict(X_doc_test)
doc_cm = confusion_matrix(y_test, doc_test_preds)

report = classification_report(y_test, best_preds, target_names=["Real", "Fake"], output_dict=True, zero_division=0)
doc_report = classification_report(y_test, doc_test_preds, target_names=["Real", "Fake"], output_dict=True, zero_division=0)

final_data = prepare_dataframe(
    pd.concat([prepared, make_document_style_records(prepared)], ignore_index=True)
)
final_tfidf, final_ohe, final_scaler, X_final = fit_feature_transformers(final_data)
final_model = clone(candidate_models[best_name])
final_model.fit(X_final, final_data[TARGET_COLUMN])

joblib.dump(final_model, MODEL_PATH)
joblib.dump(final_tfidf, TFIDF_PATH)
joblib.dump(final_ohe, LABEL_ENCODERS_PATH)
joblib.dump(final_scaler, FEATURE_CONFIG_PATH)

metrics = {
    "best_model": best_name,
    "results": results,
    "confusion_matrix": cm.tolist(),
    "document_style_confusion_matrix": doc_cm.tolist(),
    "classification_report": report,
    "document_style_classification_report": doc_report,
    "dataset_sources": dataset_sources,
    "dataset_info": {
        "total_samples": len(prepared),
        "training_samples": len(train_augmented),
        "test_samples": len(test_df),
        "final_training_samples": len(final_data),
        "feature_count": X_train.shape[1],
    },
}

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

with open(REPORT_DIR / "clustering_result.json", "w", encoding="utf-8") as f:
    json.dump(clustering_result, f, indent=2)

print(f"\nSaved model artifacts to: {MODEL_DIR}")
print(f"Saved report artifacts to: {REPORT_DIR}")

## 7. Live Inference / Prediction

The final section converts the Streamlit prediction workflow into a clean notebook function. The function accepts a raw job-posting text string, builds a project-compatible input row, applies the same preprocessing and feature transformers used during training, predicts whether the job is real or fake, estimates a fake-risk score, and prints warning indicators that help explain suspicious patterns to an examiner.

In [ ]:
def estimate_fake_probability(model, features, prediction):
    """Return a usable fake-risk score for models with or without predict_proba."""
    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(features)[0]
        classes = list(getattr(model, "classes_", [0, 1]))
        if 1 in classes:
            return float(probabilities[classes.index(1)])
        return float(probabilities[-1])

    if hasattr(model, "decision_function"):
        margin = float(model.decision_function(features)[0])
        margin = np.clip(margin, -20, 20)
        return float(1 / (1 + np.exp(-margin)))

    return 0.70 if prediction == 1 else 0.30


def risk_level(score):
    if score <= 30:
        return "Low Risk"
    if score <= 70:
        return "Medium Risk"
    return "High Risk"


def warning_indicators(prepared_row):
    row = prepared_row.iloc[0]
    warnings = []
    if row.get("profile_missing", 0) == 1:
        warnings.append("Company profile is missing")
    if row.get("has_company_logo", 0) == 0:
        warnings.append("Company logo is missing")
    if row.get("has_questions", 0) == 0:
        warnings.append("Screening questions are missing")
    if row.get("suspicious_keyword_count", 0) >= 1:
        warnings.append("Suspicious keywords found")
    if row.get("fee_keyword_count", 0) >= 1:
        warnings.append("Payment, deposit, or fee request found")
    if row.get("urgency_keyword_count", 0) >= 1:
        warnings.append("Urgency or guaranteed-selection language found")
    if row.get("contact_risk_keyword_count", 0) >= 1:
        warnings.append("Unprofessional contact channel or email pattern found")
    if row.get("sensitive_info_keyword_count", 0) >= 1:
        warnings.append("Sensitive document or bank-detail request found")
    if row.get("salary_missing", 0) == 1:
        warnings.append("Salary information is missing")
    if row.get("requirements_length", 0) < 5:
        warnings.append("Requirements are too brief or missing")
    return warnings or ["No strong suspicious indicator found"]


def predict_job_from_text(
    raw_text: str,
    title: str = "Raw Job Posting",
    employment_type: str = "Unknown",
    required_experience: str = "Unknown",
    required_education: str = "Unknown",
    industry: str = "Unknown",
    function: str = "Unknown",
    telecommuting: int = 0,
    has_company_logo: int = 0,
    has_questions: int = 0,
):
    """Predict whether a raw job-posting text appears real or fake."""
    record = build_user_record(
        title=title,
        company_profile="",
        description=raw_text,
        requirements="",
        benefits="",
        employment_type=employment_type,
        required_experience=required_experience,
        required_education=required_education,
        industry=industry,
        function=function,
        telecommuting=telecommuting,
        has_company_logo=has_company_logo,
        has_questions=has_questions,
    )

    prepared_record = prepare_dataframe(record)
    features = transform_features(prepared_record, final_tfidf, final_ohe, final_scaler)

    prediction = int(final_model.predict(features)[0])
    probability = estimate_fake_probability(final_model, features, prediction)
    score = round(probability * 100, 2)
    label = "Fake / Suspicious" if prediction == 1 else "Real / Genuine"
    level = risk_level(score)
    warnings = warning_indicators(prepared_record)

    print(f"Prediction : {label}")
    print(f"Risk Score : {score}/100")
    print(f"Risk Level : {level}")
    print("\nWarning Indicators:")
    for item in warnings:
        print(f"- {item}")

    return {
        "prediction": label,
        "risk_score": score,
        "risk_level": level,
        "warnings": warnings,
    }


sample_job_posting = """
Online Data Entry Operator. Work from home and earn Rs. 60,000 per month.
No experience required. Immediate joining. Limited seats available.
Registration fee required for training material. Contact on WhatsApp and send Aadhaar and bank details.
"""

prediction_result = predict_job_from_text(
    sample_job_posting,
    title="Online Data Entry Operator",
    employment_type="Part-time",
    required_experience="Entry level",
    industry="Internet",
    function="Administrative",
    telecommuting=1,
    has_company_logo=0,
    has_questions=0,
)